# 02. Compact MPTSNet weighted OOF + holdout Test

30봉 자금 유입 sequence로 보수적 `min_bid` net +3% label을 분류한다.

- 모델 근거: [MPTSNet, AAAI 2025](https://ojs.aaai.org/index.php/AAAI/article/view/34155)
- 공식 구현: [MUYang99/MPTSNet](https://github.com/MUYang99/MPTSNet)
- 이 notebook은 30 step·9거래일 데이터에 맞춰 채널과 block을 줄인 **compact adaptation**이다. 논문 benchmark의 원형 모델을 그대로 재현한 것은 아니다.
- Train fold에서만 robust scaling과 positive class weight를 계산한다.
- 마지막 거래일은 threshold와 epoch 선택에 사용하지 않는 최종 Test다.
- 동일 종목은 포지션 청산 전 재진입하지 않는다. TP는 +3%, 미달은 5분 timeout 수익률로 평가한다.

In [1]:
from pathlib import Path
from copy import deepcopy
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


SEED = 20260722
PREPROCESS_VERSION = 'scalp_30m_ohlcv_net3_minbid_5m_v1'
MODEL_VERSION = 'compact_mptsnet_weighted_net3_minbid_v1'
BATCH_SIZE = 512
MAX_EPOCHS = 30
PATIENCE = 6
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
MIN_OOF_TRADES = 30
STAKE_USD = 100.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

schema = json.loads((PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_schema.json').read_text(encoding='utf-8'))
metadata = pd.read_parquet(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_features.npz') as loaded:
    all_features = loaded['sequence']
feature_indices = [schema['feature_names'].index(name) for name in schema['default_feature_names']]
X = np.ascontiguousarray(all_features[:, :, feature_indices], dtype=np.float32)
del all_features
y = metadata['target_net3_within_5m'].to_numpy(np.float32)
sessions = sorted(metadata['session'].unique())
TEST_SESSION = sessions[-1]
OOF_VALID_SESSIONS = sessions[-4:-1]

assert X.shape == (len(metadata), schema['sequence_length'], len(schema['default_feature_names']))
assert np.isfinite(X).all()
print('device:', DEVICE)
print('X:', X.shape, '| positive:', f'{y.mean():.2%}')
print('OOF validation:', OOF_VALID_SESSIONS)
print('final Test:', TEST_SESSION)

device: cuda
X: (72181, 30, 38) | positive: 3.94%
OOF validation: ['session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16']
final Test: session_2026-07-17


## Train-only scaling과 주기 탐색

절대 거래량은 제외한다. scaler는 각 Train fold의 일부 표본에서 median/IQR로 계산하고 모든 입력을 ±10으로 제한한다. MPTSNet의 주기는 Train sequence FFT로만 선택한다.

In [2]:
def fit_robust_scaler(x, train_indices, max_windows=20_000):
    rng = np.random.default_rng(SEED + len(train_indices))
    chosen = np.asarray(train_indices)
    if len(chosen) > max_windows:
        chosen = rng.choice(chosen, size=max_windows, replace=False)
    flat = x[chosen].reshape(-1, x.shape[-1])
    center = np.median(flat, axis=0).astype(np.float32)
    q25, q75 = np.percentile(flat, [25, 75], axis=0)
    scale = np.maximum((q75 - q25).astype(np.float32), 1e-4)
    return center, scale


def apply_scaler(x, center, scale):
    return np.clip((x - center[None, None, :]) / scale[None, None, :], -10, 10).astype(np.float32)


def discover_periods(x_scaled, train_indices, top_k=3, max_windows=12_000):
    rng = np.random.default_rng(SEED + 17 + len(train_indices))
    chosen = np.asarray(train_indices)
    if len(chosen) > max_windows:
        chosen = rng.choice(chosen, size=max_windows, replace=False)
    spectrum = np.abs(np.fft.rfft(x_scaled[chosen], axis=1)).mean(axis=(0, 2))
    periods = []
    for frequency_index in np.argsort(spectrum[1:])[::-1] + 1:
        period = int(round(x_scaled.shape[1] / frequency_index))
        if 2 <= period <= x_scaled.shape[1] // 2 and period not in periods:
            periods.append(period)
        if len(periods) == top_k:
            break
    for fallback in [2, 3, 5, 6, 10, 15]:
        if len(periods) == top_k:
            break
        if fallback not in periods:
            periods.append(fallback)
    return periods


class SequenceDataset(Dataset):
    def __init__(self, x, targets, indices):
        self.x = x
        self.targets = targets
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]
        return torch.from_numpy(self.x[index]), torch.tensor(self.targets[index]), index

## Compact MPTSNet

시간축 attention과 FFT로 찾은 여러 period의 local convolution·chunk attention을 함께 사용한다. 원형의 매우 큰 1024-channel block 대신 `d_model=64`로 제한해 현재 데이터 크기에 맞춘다.

In [3]:
class InceptionLocal(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(d_model, d_model, kernel_size=kernel, padding=kernel // 2, groups=d_model),
                nn.Conv1d(d_model, d_model, kernel_size=1),
                nn.GELU(),
            ) for kernel in (1, 3, 5)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(torch.stack([branch(x) for branch in self.branches], dim=0).mean(dim=0))


class PeriodBranch(nn.Module):
    def __init__(self, period, d_model, n_heads, dropout):
        super().__init__()
        self.period = period
        self.local = InceptionLocal(d_model, dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 3,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.chunk_encoder = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, x):
        batch, length, channels = x.shape
        padded_length = math.ceil(length / self.period) * self.period
        if padded_length > length:
            padding = x.new_zeros(batch, padded_length - length, channels)
            x = torch.cat([x, padding], dim=1)
        chunks = x.reshape(batch, padded_length // self.period, self.period, channels)
        local = chunks.reshape(-1, self.period, channels).transpose(1, 2)
        local = (local + self.local(local)).transpose(1, 2)
        local = local.reshape(batch, -1, self.period, channels)
        chunk_context = self.chunk_encoder(local.mean(dim=2))
        mixed = local + chunk_context.unsqueeze(2)
        return mixed.reshape(batch, padded_length, channels)[:, :length]


class PeriodicMixerBlock(nn.Module):
    def __init__(self, periods, seq_len, d_model=64, n_heads=4, dropout=0.15):
        super().__init__()
        self.periods = periods
        self.seq_len = seq_len
        self.frequency_indices = [max(1, int(round(seq_len / period))) for period in periods]
        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 3,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True,
        )
        self.temporal = nn.TransformerEncoder(temporal_layer, num_layers=1)
        self.period_branches = nn.ModuleList([
            PeriodBranch(period, d_model, n_heads, dropout) for period in periods
        ])
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        temporal = self.temporal(x)
        spectrum = torch.abs(torch.fft.rfft(x.mean(dim=-1), dim=1))
        amplitudes = torch.stack([spectrum[:, min(i, spectrum.shape[1] - 1)] for i in self.frequency_indices], dim=1)
        weights = torch.softmax(amplitudes, dim=1)
        periodic = torch.stack([branch(x) for branch in self.period_branches], dim=1)
        periodic = (periodic * weights[:, :, None, None]).sum(dim=1)
        return self.norm(x + self.dropout(temporal + periodic))


class CompactMPTSNet(nn.Module):
    def __init__(self, input_dim, seq_len, periods, d_model=64, n_heads=4, blocks=2, dropout=0.15):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model))
        self.position = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.blocks = nn.ModuleList([
            PeriodicMixerBlock(periods, seq_len, d_model, n_heads, dropout) for _ in range(blocks)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * 3), nn.Linear(d_model * 3, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, 1),
        )

    def forward(self, x):
        x = self.input_projection(x) + self.position
        for block in self.blocks:
            x = block(x)
        pooled = torch.cat([x.mean(dim=1), x.amax(dim=1), x[:, -1]], dim=1)
        return self.head(pooled).squeeze(1)

## Weighted training과 expanding OOF

`pos_weight = negative / positive`를 각 Train fold에서 계산한다. early stopping은 validation PR-AUC만 사용하며 Test는 보지 않는다.

In [4]:
@torch.no_grad()
def predict_scores(model, x_scaled, indices):
    loader = DataLoader(SequenceDataset(x_scaled, y, indices), batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    probabilities, ordered_indices = [], []
    for batch_x, _, batch_indices in loader:
        logits = model(batch_x.to(DEVICE, non_blocking=True))
        probabilities.append(torch.sigmoid(logits).cpu().numpy())
        ordered_indices.append(batch_indices.numpy())
    return np.concatenate(ordered_indices), np.concatenate(probabilities)


def classification_metrics(target, probability):
    return {
        'samples': int(len(target)),
        'positives': int(np.sum(target)),
        'prevalence': float(np.mean(target)),
        'pr_auc': float(average_precision_score(target, probability)),
        'roc_auc': float(roc_auc_score(target, probability)),
        'brier': float(brier_score_loss(target, probability)),
    }


def fit_fold(x_scaled, train_indices, valid_indices, periods, fold_seed):
    torch.manual_seed(fold_seed)
    model = CompactMPTSNet(x_scaled.shape[-1], x_scaled.shape[1], periods).to(DEVICE)
    train_target = y[train_indices]
    positives = float(train_target.sum())
    negatives = float(len(train_target) - positives)
    pos_weight_value = negatives / max(positives, 1.0)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=3e-5)
    generator = torch.Generator().manual_seed(fold_seed)
    loader = DataLoader(
        SequenceDataset(x_scaled, y, train_indices), batch_size=BATCH_SIZE, shuffle=True,
        generator=generator, pin_memory=DEVICE.type == 'cuda',
    )
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        for batch_x, batch_y, _ in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(batch_x)
        _, valid_probability = predict_scores(model, x_scaled, valid_indices)
        valid_pr_auc = average_precision_score(y[valid_indices], valid_probability)
        history.append({
            'epoch': epoch, 'train_loss': loss_sum / len(train_indices),
            'valid_pr_auc': float(valid_pr_auc), 'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if valid_pr_auc > best_score + 1e-4:
            best_score = valid_pr_auc
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        scheduler.step()
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), pos_weight_value, best_epoch


fold_metrics = []
fold_histories = []
oof_frames = []
final_bundle = None
started = time.time()

for fold_number, valid_session in enumerate(OOF_VALID_SESSIONS, start=1):
    train_sessions = [session for session in sessions if session < valid_session]
    train_indices = np.flatnonzero(metadata['session'].isin(train_sessions).to_numpy())
    valid_indices = np.flatnonzero(metadata['session'].eq(valid_session).to_numpy())
    center, scale = fit_robust_scaler(X, train_indices)
    X_scaled = apply_scaler(X, center, scale)
    periods = discover_periods(X_scaled, train_indices)
    model, history, pos_weight, best_epoch = fit_fold(
        X_scaled, train_indices, valid_indices, periods, SEED + fold_number,
    )
    train_order, train_probability = predict_scores(model, X_scaled, train_indices)
    valid_order, valid_probability = predict_scores(model, X_scaled, valid_indices)
    train_result = classification_metrics(y[train_order], train_probability)
    valid_result = classification_metrics(y[valid_order], valid_probability)
    fold_metrics.append({
        'fold': fold_number, 'valid_session': valid_session, 'train_sessions': train_sessions,
        'periods': periods, 'pos_weight': pos_weight, 'best_epoch': best_epoch,
        **{f'train_{key}': value for key, value in train_result.items()},
        **{f'valid_{key}': value for key, value in valid_result.items()},
    })
    history['fold'] = fold_number
    history['valid_session'] = valid_session
    fold_histories.append(history)
    fold_frame = metadata.iloc[valid_order].copy()
    fold_frame['fold'] = fold_number
    fold_frame['probability'] = valid_probability
    oof_frames.append(fold_frame)
    if valid_session == sessions[-2]:
        final_bundle = {
            'state_dict': {name: value.detach().cpu().clone() for name, value in model.state_dict().items()},
            'center': center.copy(), 'scale': scale.copy(), 'periods': periods,
            'train_sessions': train_sessions, 'calibration_session': valid_session,
            'pos_weight': pos_weight, 'best_epoch': best_epoch,
        }
    print(
        f"fold {fold_number} | val={valid_session} | periods={periods} | weight={pos_weight:.1f} | "
        f"epoch={best_epoch} | train AP={train_result['pr_auc']:.4f} | val AP={valid_result['pr_auc']:.4f}"
    )
    del model, X_scaled
    torch.cuda.empty_cache()
    gc.collect()

fold_metrics = pd.DataFrame(fold_metrics)
training_history = pd.concat(fold_histories, ignore_index=True)
oof = pd.concat(oof_frames).sort_values('entry_timestamp').reset_index(drop=True)
print(f'OOF elapsed: {(time.time() - started) / 60:.2f} min')
display(fold_metrics[['fold', 'valid_session', 'periods', 'pos_weight', 'best_epoch', 'train_pr_auc', 'valid_pr_auc', 'valid_prevalence']])

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 1 | val=session_2026-07-14 | periods=[15, 10, 8] | weight=22.9 | epoch=3 | train AP=0.1452 | val AP=0.1250


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 2 | val=session_2026-07-15 | periods=[15, 10, 8] | weight=23.2 | epoch=4 | train AP=0.1439 | val AP=0.1365


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 3 | val=session_2026-07-16 | periods=[15, 10, 8] | weight=23.1 | epoch=7 | train AP=0.1870 | val AP=0.0949
OOF elapsed: 1.53 min


,fold,valid_session,periods,pos_weight,best_epoch,train_pr_auc,valid_pr_auc,valid_prevalence
0,1,session_2026-07-14,"[15, 10, 8]",22.866408,3,0.145185,0.125034,0.037768
1,2,session_2026-07-15,"[15, 10, 8]",23.201932,4,0.143918,0.136494,0.042800
2,3,session_2026-07-16,"[15, 10, 8]",23.063735,7,0.186999,0.094877,0.040047


## OOF 진입 비율 선택

가중 loss의 확률은 그대로 0.5를 쓰지 않는다. 각 OOF fold의 상위 score 비율을 비교하고, 최소 30건·2일 이상 거래 조건에서 평균 수익률의 표준오차를 차감한 값을 최대화한다.

In [5]:
def cooldown_trade_frame(frame, signal):
    work = frame.copy()
    work['_signal'] = np.asarray(signal, dtype=bool)
    selected = []
    for _, group in work.sort_values('entry_timestamp').groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.iterrows():
            if not row['_signal']:
                continue
            if available_at is not None and row['entry_timestamp'] < available_at:
                continue
            selected.append(index)
            hold_minutes = int(row['first_hit_minute']) if row['target_net3_within_5m'] == 1 else 5
            available_at = row['entry_timestamp'] + pd.Timedelta(minutes=hold_minutes)
    trades = work.loc[selected].sort_values('entry_timestamp').copy()
    trades['trade_return'] = np.where(
        trades['target_net3_within_5m'].eq(1), 0.03, trades['timeout_net_return_5m'],
    )
    trades['pnl_usd'] = STAKE_USD * trades['trade_return']
    return trades


def backtest_metrics(frame, signal):
    trades = cooldown_trade_frame(frame, signal)
    if len(trades) == 0:
        return {key: np.nan for key in [
            'precision', 'recall', 'win_rate', 'mean_return', 'median_return',
            'risk_adjusted_return', 'total_pnl_usd', 'profit_factor', 'max_drawdown_usd',
        ]} | {'trades': 0, 'sessions_traded': 0}
    returns = trades['trade_return'].to_numpy()
    pnl = trades['pnl_usd'].to_numpy()
    equity = np.cumsum(pnl)
    peak = np.maximum.accumulate(np.r_[0.0, equity])
    drawdown = np.r_[0.0, equity] - peak
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        'trades': int(len(trades)),
        'sessions_traded': int(trades['session'].nunique()),
        'precision': float(trades['target_net3_within_5m'].mean()),
        'recall': float(trades['target_net3_within_5m'].sum() / max(frame['target_net3_within_5m'].sum(), 1)),
        'win_rate': float((returns > 0).mean()),
        'mean_return': float(returns.mean()),
        'median_return': float(np.median(returns)),
        'risk_adjusted_return': float(returns.mean() - returns.std(ddof=1) / math.sqrt(len(returns))) if len(returns) > 1 else float(returns.mean()),
        'total_pnl_usd': float(pnl.sum()),
        'profit_factor': float(gross_profit / gross_loss) if gross_loss > 0 else float('inf'),
        'max_drawdown_usd': float(drawdown.min()),
    }


TOP_FRACTIONS = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20]
candidate_rows = []
for top_fraction in TOP_FRACTIONS:
    signal = np.zeros(len(oof), dtype=bool)
    thresholds = []
    for fold, group in oof.groupby('fold'):
        threshold = float(group['probability'].quantile(1 - top_fraction))
        thresholds.append(threshold)
        signal[group.index] = group['probability'].to_numpy() >= threshold
    candidate_rows.append({
        'top_fraction': top_fraction, 'mean_fold_threshold': float(np.mean(thresholds)),
        **backtest_metrics(oof, signal),
    })
candidate_table = pd.DataFrame(candidate_rows)
eligible = candidate_table[
    candidate_table['trades'].ge(MIN_OOF_TRADES) & candidate_table['sessions_traded'].ge(2)
]
if eligible.empty:
    eligible = candidate_table[candidate_table['trades'].gt(0)]
selected = eligible.sort_values(['risk_adjusted_return', 'total_pnl_usd']).iloc[-1]
SELECTED_TOP_FRACTION = float(selected['top_fraction'])
DEPLOYMENT_ELIGIBLE = bool(
    selected['risk_adjusted_return'] > 0
    and selected['total_pnl_usd'] > 0
    and selected['profit_factor'] > 1
)
print('selected top fraction:', f'{SELECTED_TOP_FRACTION:.2%}')
print('deployment eligible:', DEPLOYMENT_ELIGIBLE)
display(candidate_table.style.format({
    'top_fraction': '{:.2%}', 'precision': '{:.2%}', 'recall': '{:.2%}', 'win_rate': '{:.2%}',
    'mean_return': '{:.3%}', 'median_return': '{:.3%}', 'risk_adjusted_return': '{:.3%}',
    'total_pnl_usd': '${:.2f}', 'max_drawdown_usd': '${:.2f}',
}))

selected top fraction: 20.00%
deployment eligible: False


,top_fraction,mean_fold_threshold,trades,sessions_traded,precision,recall,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd
0,0.10%,0.860771,12,3,25.00%,0.33%,41.67%,-2.932%,-1.770%,-4.716%,$-35.18,0.253355,$-35.79
1,0.25%,0.853398,28,3,28.57%,0.88%,39.29%,-1.879%,-1.462%,-2.784%,$-52.62,0.344881,$-57.33
2,0.50%,0.845229,55,3,27.27%,1.64%,38.18%,-1.838%,-1.229%,-2.434%,$-101.12,0.335663,$-101.74
3,1.00%,0.835014,101,3,16.83%,1.86%,35.64%,-1.734%,-1.229%,-2.158%,$-175.17,0.331656,$-182.66
4,2.00%,0.822347,178,3,14.04%,2.74%,35.96%,-1.885%,-1.258%,-2.222%,$-335.56,0.308784,$-338.79
5,3.00%,0.810785,245,3,12.24%,3.29%,33.47%,-1.820%,-1.533%,-2.091%,$-445.93,0.312511,$-448.93
6,5.00%,0.792731,361,3,12.19%,4.82%,32.69%,-1.647%,-1.603%,-1.858%,$-594.48,0.341682,$-601.27
7,7.50%,0.773483,509,3,13.36%,7.46%,33.99%,-1.550%,-1.329%,-1.731%,$-788.79,0.373368,$-798.35
8,10.00%,0.753063,635,3,11.34%,7.89%,33.86%,-1.594%,-1.174%,-1.762%,$-1012.04,0.348039,$-1023.52
9,15.00%,0.714063,891,3,11.67%,11.40%,33.56%,-1.547%,-1.250%,-1.685%,$-1378.77,0.344633,$-1388.96


## 고정 threshold 최종 Test

최종 모델은 7월 16일 validation fold에서 선택된 checkpoint다. 7월 16일 score의 선택 percentile을 숫자 threshold로 고정한 후 7월 17일 Test에 그대로 적용한다.

In [6]:
calibration_oof = oof[oof['session'].eq(final_bundle['calibration_session'])]
FIXED_THRESHOLD = float(calibration_oof['probability'].quantile(1 - SELECTED_TOP_FRACTION))
test_indices = np.flatnonzero(metadata['session'].eq(TEST_SESSION).to_numpy())
X_final_scaled = apply_scaler(X, final_bundle['center'], final_bundle['scale'])
final_model = CompactMPTSNet(
    X.shape[-1], X.shape[1], final_bundle['periods'],
).to(DEVICE)
final_model.load_state_dict(final_bundle['state_dict'])
test_order, test_probability = predict_scores(final_model, X_final_scaled, test_indices)
test_predictions = metadata.iloc[test_order].copy().reset_index(drop=True)
test_predictions['probability'] = test_probability
test_predictions['signal'] = test_predictions['probability'].ge(FIXED_THRESHOLD)
test_classification = classification_metrics(
    test_predictions['target_net3_within_5m'].to_numpy(), test_predictions['probability'].to_numpy(),
)
test_backtest = backtest_metrics(test_predictions, test_predictions['signal'].to_numpy())
test_trades = cooldown_trade_frame(test_predictions, test_predictions['signal'].to_numpy())
oof_classification = classification_metrics(
    oof['target_net3_within_5m'].to_numpy(), oof['probability'].to_numpy(),
)

model_path = MODEL_ROOT / f'{MODEL_VERSION}.pt'
oof_path = RESULT_ROOT / f'{MODEL_VERSION}_oof_predictions.parquet'
test_path = RESULT_ROOT / f'{MODEL_VERSION}_test_predictions.parquet'
trades_path = RESULT_ROOT / f'{MODEL_VERSION}_test_trades.parquet'
fold_path = RESULT_ROOT / f'{MODEL_VERSION}_fold_metrics.parquet'
candidate_path = RESULT_ROOT / f'{MODEL_VERSION}_threshold_candidates.parquet'
history_path = RESULT_ROOT / f'{MODEL_VERSION}_training_history.parquet'
metrics_path = RESULT_ROOT / f'{MODEL_VERSION}_metrics.json'

torch.save({
    'model_version': MODEL_VERSION, 'architecture': 'CompactMPTSNet',
    'state_dict': final_bundle['state_dict'], 'input_dim': X.shape[-1], 'seq_len': X.shape[1],
    'periods': final_bundle['periods'], 'feature_names': schema['default_feature_names'],
    'scaler_center': torch.from_numpy(final_bundle['center'].copy()),
    'scaler_scale': torch.from_numpy(final_bundle['scale'].copy()),
    'train_sessions': final_bundle['train_sessions'],
    'calibration_session': final_bundle['calibration_session'], 'test_session': TEST_SESSION,
    'pos_weight': final_bundle['pos_weight'], 'best_epoch': final_bundle['best_epoch'],
    'selected_top_fraction': SELECTED_TOP_FRACTION, 'fixed_threshold': FIXED_THRESHOLD,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}, model_path)
oof.to_parquet(oof_path, index=False, compression='zstd')
test_predictions.to_parquet(test_path, index=False, compression='zstd')
test_trades.to_parquet(trades_path, index=False, compression='zstd')
fold_metrics.to_parquet(fold_path, index=False, compression='zstd')
candidate_table.to_parquet(candidate_path, index=False, compression='zstd')
training_history.to_parquet(history_path, index=False, compression='zstd')
metrics = {
    'model_version': MODEL_VERSION, 'model_parameters': sum(p.numel() for p in final_model.parameters()),
    'preprocess_version': PREPROCESS_VERSION, 'oof_valid_sessions': OOF_VALID_SESSIONS,
    'train_sessions': final_bundle['train_sessions'],
    'calibration_session': final_bundle['calibration_session'], 'test_session': TEST_SESSION,
    'periods': final_bundle['periods'], 'pos_weight': final_bundle['pos_weight'],
    'best_epoch': final_bundle['best_epoch'], 'selected_top_fraction': SELECTED_TOP_FRACTION,
    'fixed_threshold': FIXED_THRESHOLD, 'deployment_eligible': DEPLOYMENT_ELIGIBLE,
    'oof_classification': oof_classification,
    'selected_oof_backtest': {key: (None if pd.isna(value) else float(value)) for key, value in selected.to_dict().items()},
    'test_classification': test_classification, 'test_backtest': test_backtest,
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

summary = pd.DataFrame([
    {'split': 'OOF', **oof_classification, **{f'bt_{k}': v for k, v in selected.to_dict().items()}},
    {'split': 'Test', **test_classification, **{f'bt_{k}': v for k, v in test_backtest.items()}},
])
print('parameters:', f"{metrics['model_parameters']:,}")
print('fixed threshold:', f'{FIXED_THRESHOLD:.6f}')
print('deployment eligible:', DEPLOYMENT_ELIGIBLE)
print('model:', model_path)
display(summary.T)
display(test_trades[[
    'entry_timestamp', 'symbol', 'probability', 'target_net3_within_5m',
    'first_hit_minute', 'trade_return', 'pnl_usd',
]])

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


parameters: 430,913
fixed threshold: 0.644367
deployment eligible: False
model: /home/user/urbandatalab/YSLee/model/compact_mptsnet_weighted_net3_minbid_v1.pt


,0,1
split,OOF,Test
samples,22517,6609
positives,912,131
prevalence,0.040503,0.019821
pr_auc,0.118473,0.049531
roc_auc,0.781531,0.744565
brier,0.202124,0.141721
bt_top_fraction,0.2,NaN
bt_mean_fold_threshold,0.670762,NaN
bt_trades,1144.0,293.0


,entry_timestamp,symbol,probability,target_net3_within_5m,first_hit_minute,trade_return,pnl_usd
3158,2026-07-17 09:10:00+00:00,GCTK,0.730713,0,0,0.050899,5.089903
3163,2026-07-17 09:15:00+00:00,GCTK,0.846547,0,0,-0.045374,-4.537377
3168,2026-07-17 09:20:00+00:00,GCTK,0.822053,0,0,0.014453,1.445332
3173,2026-07-17 09:25:00+00:00,GCTK,0.832795,0,0,-0.032120,-3.212006
3178,2026-07-17 09:30:00+00:00,GCTK,0.657443,0,0,-0.055536,-5.553588
...,...,...,...,...,...,...,...
71,2026-07-17 22:44:00+00:00,ADVB,0.700064,0,0,-0.029807,-2.980659
4378,2026-07-17 22:47:00+00:00,NXXT,0.687052,0,0,-0.026570,-2.657008
6412,2026-07-17 22:47:00+00:00,STAK,0.650134,0,0,-0.031356,-3.135641
6608,2026-07-17 22:47:00+00:00,VMAR,0.752345,0,0,-0.019411,-1.941059
